# Zero-shot baselines (dev)

Generation runs on Google Colab (T4); scoring runs locally. Predictions already generated are in `preds/`.

Run the generation cells in order: Setup, then one model section.

## Setup (Colab)

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # Colab secret HF_TOKEN

In [ ]:
!git clone https://github.com/mohinska/mmculture-qa.git
%cd mmculture-qa
!pip install -q transformers==4.57.6 accelerate pillow bitsandbytes qwen-omni-utils torchvision python-dotenv tqdm huggingface_hub
%env MMCQA_OUT=/content

In [ ]:
!python shared/download_data.py dev --task qa

## Qwen2.5-Omni-3B (official reference baseline)

Full dev (1000 items), 4-bit. Resumable: rerun the cell after a crash.

In [ ]:
!python t2_text/run_vlm.py --model Qwen/Qwen2.5-Omni-3B --prompt-id official --n 0 --load-4bit --dtype float16

## Gemma 3 4B

Accept the license on the Hugging Face model page first. fp16 gave empty answers on a T4, so bfloat16 is the first try and `--load-4bit` is the fallback.

In [ ]:
!python t2_text/run_vlm.py --model google/gemma-3-4b-it --prompt-id plain --n 200 --dtype bfloat16

In [ ]:
!python t2_text/run_vlm.py --model google/gemma-3-4b-it --prompt-id gold_style_v1 --n 200 --dtype bfloat16

## Qwen3-VL-4B and 8B

Answers already exist, so these cells are commented out.

In [ ]:
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-4B-Instruct --prompt-id plain --n 200
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-4B-Instruct --prompt-id gold_style_v1 --n 200

In [ ]:
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-8B-Instruct --prompt-id plain --n 200
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-8B-Instruct --prompt-id gold_style_v1 --n 200

## Download

In [ ]:
!zip -r /content/preds.zip /content/preds
from google.colab import files
files.download("/content/preds.zip")
# or copy to Drive:
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/preds.zip /content/drive/MyDrive/

## Scoring

Runs locally, not on Colab. Prerequisites:

- the project venv with `pip install -r requirements.txt` (Apple Silicon Mac or Linux)
- predictions in `preds/` (`t2_text_dev_en_*.jsonl`, `{id, answer, lang}` rows)

`evaluate()` from `shared/metrics.py` is the only scoring path. It writes `results/<name>.json` and `preds/<name>.scores.jsonl`. The first run downloads about 1 GB of mmBERT weights.

In [ ]:
import json, subprocess, sys, tempfile
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]  # notebook lives in t2_text/notebooks/
sys.path.insert(0, str(REPO / "shared"))
sys.path.insert(0, str(REPO / "t2_text"))
from data import PREDS_DIR, RESULTS_DIR, load_split, load_jsonl
from metrics import evaluate
from prompts import PROMPTS

import pandas as pd

gold = {r["id"]: r["answer"] for r in load_split("qa", "dev")}

In [ ]:
# preds files with at least one non-empty answer
files = []
for f in sorted(PREDS_DIR.glob("t2_text_dev_en_*.jsonl")):
    if f.name.endswith(".scores.jsonl"):
        continue
    if not any(str(r.get("answer") or "").strip() for r in load_jsonl(f)):
        print("skip (all answers empty):", f.name)
        continue
    files.append(f)
print(len(files), "files to score")

In [ ]:
for f in files:
    print("scoring", f.name)
    try:
        evaluate(f, "dev")
    except Exception as e:
        print("FAILED:", f.name, type(e).__name__, e)

In [ ]:
def split_stem(stem):
    """t2_text_dev_en_<model>_<prompt> -> (model, prompt); prompt is a key of PROMPTS."""
    tail = stem.removeprefix("t2_text_dev_en_")
    for prompt in PROMPTS:
        if tail.endswith("_" + prompt):
            return tail[:-len(prompt) - 1], prompt
    return tail, "?"

table = []
for f in files:
    out = RESULTS_DIR / f"{f.stem}.json"
    if not out.exists():
        continue
    r = json.loads(out.read_text(encoding="utf-8"))
    s = r["submission_result"]
    model, prompt = split_stem(f.stem)
    table.append({"model": model, "prompt": prompt, "n": r["n"],
                  "BERTScore-F1": s["BERTScore-F1"], "P": s["BERTScore-P"], "R": s["BERTScore-R"],
                  "BLEU": s["BLEU"], "ROUGE-L": s["ROUGE-L"], "coverage": s["Coverage"]})
pd.DataFrame(table).sort_values("BERTScore-F1", ascending=False).reset_index(drop=True)

In [ ]:
def med_words(texts):
    return pd.Series([len(str(t).split()) for t in texts]).median()

for f in files:
    rows = load_jsonl(f)
    print(f"{f.stem.removeprefix('t2_text_dev_en_'):45s} pred median {med_words(r['answer'] for r in rows):5.1f}"
          f"  gold median {med_words(gold[r['id']] for r in rows):5.1f}")

### Optional cross-check (Task 7)

Runs the standalone `score.py` on the first scored file and compares its BERTScore-F1 with `evaluate()` to 4 decimals. Not part of the normal flow.

In [ ]:
f = files[0]
ids = {r["id"] for r in load_jsonl(f)}
with tempfile.TemporaryDirectory() as tmp:
    g, o = Path(tmp) / "dev_en.jsonl", Path(tmp) / "out.json"  # score.py reads the track from the gold file name
    g.write_text("".join(json.dumps({"id": i, "answer": gold[i]}, ensure_ascii=False) + "\n"
                         for i in sorted(ids)), encoding="utf-8")
    subprocess.run([sys.executable, "shared/official_scorer/score.py", "--gold", str(g),
                    "--pred", str(f), "--out", str(o)], check=True, cwd=REPO)
    official = json.loads(o.read_text(encoding="utf-8"))["submission_result"]["BERTScore-F1"]
ours = json.loads((RESULTS_DIR / f"{f.stem}.json").read_text(encoding="utf-8"))["bertscore_f1"]
print(f.name, "score.py:", round(official, 4), "evaluate():", round(ours, 4),
      "-> equal" if round(official, 4) == round(ours, 4) else "-> DIFFERENT")